# Predicting Air Turbulence Using Machine Learning
By: Chee Qian Wen
<br>Feb 25

## Part 3 of 5: Feature Engineering
In Part 2, the focus was on cleaning and merging all the data into a single dataset. Please refer to Part 2 to see how the following data were combined:
1) 'combined_data.csv'

Part 3 involves engineering new features based on the existing data, and creating dummy variables (one-hot encoding) for categorical features.

### 3.1. Import Libraries
First, the necessary libraries required are imported.

In [3]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

### 3.2. Feature Engineering
Feature engineering is a crucial step in building effective machine learning models. Raw data often lacks the structure needed for algorithms to make accurate predictions. By transforming, creating, or selecting meaningful features, we help models learn patterns more effectively, improve accuracy, and reduce overfitting.

In this section, several features related to time, location, distance, and wind direction will be created.

#### 3.2.1. Extract `YEAR`, `MONTH`, and `HOUR` features

In [6]:
df = pd.read_csv('combined_data.csv', low_memory=False)

In [7]:
df['DATETIME'] = pd.to_datetime(df['DATETIME'])

df['YEAR'] = df['DATETIME'].dt.year
df['MONTH'] = df['DATETIME'].dt.month
df['HOUR'] = df['DATETIME'].dt.hour

#### 3.2.2. Create `SEASON` feature based on date
Seasonality is known to affect air turbulence. Cooler or warmer air temperatures affect atmospheric conditions that directly impact air movement. 

In winter, stronger winds and stable air create more mountain wave turbulence, affecting flights over mountainous regions. The stronger temperature contrasts between the poles and the equator intensify jet streams, increasing clear-air turbulence (CAT) at cruising altitudes. Winter storms bring their own turbulence risks, especially near cold fronts and low-pressure systems. 

In summer, weaker jet streams may reduce CAT but increase convective turbulence due to thunderstorms, which generate severe updrafts, downdrafts, and wind shear, leading to turbulence.

In [9]:
# winter defined as Dec 21 to Mar 19
# spring defined as Mar 20 to Jun 20
# summer defined as Jun 21 to Sep 21
# autumn defined as Sep 22 to Dec 20

def get_season(dt):
    year = dt.year  # Get the year of the current row
    seasons = {
        'Winter': (pd.Timestamp(f'{year}-12-21'), pd.Timestamp(f'{year+1}-03-19')),
        'Spring': (pd.Timestamp(f'{year}-03-20'), pd.Timestamp(f'{year}-06-20')),
        'Summer': (pd.Timestamp(f'{year}-06-21'), pd.Timestamp(f'{year}-09-21')),
        'Autumn':   (pd.Timestamp(f'{year}-09-22'), pd.Timestamp(f'{year}-12-20'))
    }

    for season, (start, end) in seasons.items():
        if start <= dt <= end:
            return season
    return 'Winter'  # Dates before March 19 fall in Winter (previous year)

# Create 'SEASON' column
df['SEASON'] = df['DATETIME'].apply(get_season)

#### 3.2.3. Create `TIME_OF_DAY` feature based on time
The severity of turbulence is known to be affected by the time of day. During the day, sunlight heats the Earth's surface, creating thermal updrafts (rising warm air). This leads to convective turbulence, especially over land. Turbulence is thus stronger in the afternoon, when surface temperatures peak. During the night, there is less solar heating, leading to fewer updrafts and smoother flights.

In [11]:
# sunrise defined as 5am to 8.59am
# day defined as 9am to 4.59pm
# sunset defined as 5pm to 8.59pm
# night defined as 9pm to 4.49am

def get_time_of_day(dt):
    hour = dt.hour
    if 5 <= hour < 9:
        return 'Sunrise'
    elif 9 <= hour < 17:
        return 'Day'
    elif 17 <= hour < 21:
        return 'Sunset'
    else:
        return 'Night'

# Create 'TIME_OF_DAY' column
df['TIME_OF_DAY'] = df['DATETIME'].apply(get_time_of_day)

#### 3.2.4. Create `WIND_DIR` feature based on wind direction in degrees
The dataset currently reports wind direction in degrees. Wind direction is a key factor in turbulence because it influences airflow patterns, wind shear, and mechanical disturbances. This section converts wind direction in degrees to cardinal wind direction (e.g., N,S,E,W) for easier interpretation.

In [13]:
# split wind direction into 8 quadrants based on angle of wind
def degrees_to_cardinal_8(deg):
    directions = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
    index = round(deg / 45) % 8  # 360° divided into 8 sectors of 45° each
    return directions[index]

# Apply function
df['WIND_DIR'] = df['avg_wind_drct'].apply(degrees_to_cardinal_8)

#### 3.2.5. Create `DISTANCE_TO_GROUND` feature based on difference between flight level and elevation
Turbulence is usually stronger near the ground due to buildings, hills, and terrain disrupting airflow, creating chaotic wind patterns. Aircrafts are also most susceptible to heat from the ground when they are closer to the ground, leading to thermal turbulence.

In [15]:
# calculate distance to ground by subtracting ground elevation from flight level for each row
df['DISTANCE_TO_GROUND'] = df['FL'] - df['Elevation']

#### 3.2.6. Create `STATE` feature based on longitude and latitude
The dataset currently represents each location as longitude and latitude coordinates. For easier interpretation, the specific U.S. state where each incident is reported is identified.

In [17]:
url = "https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_state_500k.zip"
states = gpd.read_file(url)[['NAME', 'geometry']].to_crs(epsg=4326)

# convert the existing DataFrame to a GeoDataFrame in Chunks
chunk_size = 100000  # Adjust based on available memory
chunks = []

for i in range(0, len(df), chunk_size):
    df_chunk = df.iloc[i:i + chunk_size].copy()  # Avoid modifying original df
    
    # Convert to GeoDataFrame
    df_chunk['geometry'] = gpd.points_from_xy(df_chunk.LON.astype('float32'), df_chunk.LAT.astype('float32'))
    df_chunk = gpd.GeoDataFrame(df_chunk, geometry='geometry', crs="EPSG:4326")
    
    # Spatial join with states
    df_chunk = gpd.sjoin(df_chunk, states, how="left", predicate="within")
    
    # Keep only needed columns
    df_chunk = df_chunk[['LAT', 'LON', 'NAME']].rename(columns={'NAME': 'STATE'})
    
    chunks.append(df_chunk)  # Store processed chunk

# Combine processed chunks into a single DataFrame
state_df = pd.concat(chunks, ignore_index=True)

In [18]:
df['STATE'] = state_df['STATE'].values

#### 3.2.7. Create `TURB_DEG` based on turbulence category
This dataset currently represents turbulence as a categorical value. For analyses, this section converts it to a numerical value based on the approach used by Turbli (https://turbli.com/blog/new-us-turbulence-map-based-on-1-million-pilot-reports/). In some cases, pilots include a range of turbulence levels. In this case the average between the two bounding levels is computed. Reports adding the word “CHOP” were added a 12.5 increment. Some examples of the conversion from category to numbers are shown below:
* LGT (LIGHT) = 25
* MOD (MODERATE) = 50
* SVR (SEVERE) = 75
* EXT (EXTREME) = 100
* LGT-MOD = 37.5
* SVR CHOP = 87.5

In [20]:
# Mapping values
severity_values = {'LGT': 25, 'MOD': 50, 'SVR': 75, 'EXT': 100}

def calculate_severity(value):
    parts = value.split()  # Split by space
    has_chop = 'CHOP' in parts  # Check if CHOP is present
    numeric_values = [severity_values[p] for p in parts if p in severity_values]  # Extract valid values
    
    if numeric_values:
        avg_severity = np.mean(numeric_values)  # Compute average
    else:
        avg_severity = 0  # Default if no valid severity terms

    return avg_severity + (12.5 if has_chop else 0)  # Add CHOP penalty if present

# Apply function to new column
df['TURB_DEG'] = df['TURB_CLEANED'].apply(calculate_severity)

#### 3.2.8. Create `TURB` based on turbulence category
The current `TURB_CAT` features has too many categories. To improve the efficiency of the machine learning model, this is reduced to 3 categories: LIGHT, MODERATE, and SEVERE.

In [22]:
turb_mapping = {
    'CHOP': 'LIGHT', 'LGT': 'LIGHT', 'CHOP LGT': 'LIGHT', 'LGT CHOP': 'LIGHT', 'LGT MOD': 'LIGHT',
    'CHOP LGT MOD': 'LIGHT', 'LGT MOD SVR': 'LIGHT', 'LGT SVR': 'LIGHT', 'MOD': 'LIGHT',
    'LGT MOD EXT': 'LIGHT', 'CHOP LGT MOD SVR': 'MODERATE', 'CHOP LGT SVR': 'MODERATE',
    'CHOP MOD': 'MODERATE', 'LGT EXT': 'MODERATE', 'LGT MOD SVR EXT': 'MODERATE',
    'MOD CHOP': 'MODERATE', 'MOD SVR': 'MODERATE', 'LGT SVR EXT': 'MODERATE',
    'CHOP LGT EXT': 'MODERATE', 'CHOP MOD SVR': 'SEVERE', 'MOD EXT': 'SEVERE',
    'MOD SVR EXT': 'SEVERE', 'SVR': 'SEVERE', 'CHOP LGT SVR EXT': 'SEVERE',
    'CHOP SVR': 'SEVERE', 'CHOP MOD EXT': 'SEVERE', 'CHOP MOD SVR EXT': 'SEVERE',
    'CHOP SVR EXT': 'SEVERE', 'SVR EXT': 'SEVERE', 'EXT': 'SEVERE', 'CHOP EXT': 'SEVERE'
}

# Map values
df['TURB'] = df['TURB_CLEANED'].map(turb_mapping)

### 3.3. Data Cleaning
Since machine learning algorithms cannot handle missing values, we will drop unnecessary columns and rows with null values in the dataset.

In [24]:
df.drop(columns=['Unnamed: 0', 'AIRCRAFT', 'TURBULENCE','st_lon', 'st_lat','MODEL','station'], inplace=True)
df = df.dropna()

In [25]:
df.rename(columns={'TURB_CLEANED': 'TURB_CAT', 
                   'AIRCRAFT_TYPE': 'AC', 
                   'Elevation':'ELEV', 
                   'max_temp_c': 'MAX_TEMP', 
                   'min_temp_c' :'MIN_TEMP', 
                   'avg_wind_speed_kts':'WIND',
                   'avg_wind_drct': 'WIND_DRCT', 
                   'avg_rh':'HUM'}, inplace=True)

### 3.4. Dummy Encoding (One-hot Encoding)
In machine learning, one-hot encoding is essential because most models require numerical input and cannot directly process categorical data. One-hot encoding is a way to turn words or categories into numbers so computers can understand them. One-hot encoding creates separate columns for each category and marks them with 1s and 0s.

This section converts the categorical variables into dummy variables for each category using one-hot encoding.

In [27]:
df.rename(columns={'\u200b\u200bWTC':'WTC'}, inplace = True)

df['YEAR'] = df['YEAR'].astype(str)
df['MONTH'] = df['MONTH'].astype(str)
df['HOUR'] = df['HOUR'].astype(str)

# get dummies for categorical variables
dummies = pd.get_dummies(df[['DESCRIPTION','ENGINE_TYPE','WTC','YEAR','MONTH','HOUR','SEASON','TIME_OF_DAY','WIND_DIR','STATE']])  

# Concatenate the dummy columns with the original DataFrame  
df = pd.concat([df, dummies], axis=1)

### 3.5. Export Cleaned Data to CSV

In [29]:
df.to_csv('cleaned_data.csv')

## Proceed to Part 4.

There is now one cleaned dataset, complete with all the features for analyses, 'cleaned_data.csv'.

Part 4 explains these features are analysed in an Exploratory Data Analyses (EDA) to identify the key features for modelling.